![logo](images/untumbes.PNG)

<center><b>Prof. Dr. Jorge Zavaleta - zavaleta.jorge@gmail.com</b></center>

# Reglas de asociación

In [3]:
#librarys
import pandas as pd
import numpy as np
#
import heapq
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder
#
from surprise import Dataset, Reader, KNNBasic
# ml
from surprise.model_selection import train_test_split

# graphics
import matplotlib.pylab as plt
%matplotlib inline
#
import warnings
warnings.filterwarnings("ignore")

>## Situación a resolver

1. Una tienda que vende accesorios para teléfonos móviles realiza una promoción en placas frontales.
2. Los clientes que compren varias placas frontales entre una selección de seis colores diferentes obtienen un descuento.
3. Los gerentes de la tienda, que querían saber qué colores de placas frontales es probable que los clientes compren juntos, recopilaron la base de datos de transacciones [1]

In [4]:
# Load and preprocess data set
data = pd.read_csv('data/Faceplate.csv')
data.set_index('Transaction', inplace=True)
data.head(10)

,Red,White,Blue,Orange,Green,Yellow
Transaction,,,,,,
1,1,1,0,0,1,0
2,0,1,0,1,0,0
3,0,1,1,0,0,0
4,1,1,0,1,0,0
5,1,0,1,0,0,0
6,0,1,1,0,0,0
7,1,0,1,0,0,0
8,1,1,1,0,1,0
9,1,1,1,0,0,0


In [5]:
data.shape

(10, 6)

>## Crear las frequencias

In [6]:
# create frequent itemsets
itemsets = apriori(data, min_support=0.2, use_colnames=True)
itemsets[['itemsets','support']]

,itemsets,support
0,frozenset({Red}),0.6
1,frozenset({White}),0.7
2,frozenset({Blue}),0.6
3,frozenset({Orange}),0.2
4,frozenset({Green}),0.2
5,"frozenset({Red, White})",0.4
6,"frozenset({Red, Blue})",0.4
7,"frozenset({Red, Green})",0.2
8,"frozenset({Blue, White})",0.4
9,"frozenset({White, Orange})",0.2


>## Convertir las frecuencias en reglas

In [7]:
# convert into rules
rules = association_rules(itemsets, metric='confidence', min_threshold=0.5)
rules.sort_values(by=['lift'], ascending=False).head(6)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
13,"frozenset({Red, White})",frozenset({Green}),0.4,0.2,0.2,0.5,2.500000,0.12,1.6,1.000
15,frozenset({Green}),"frozenset({Red, White})",0.2,0.4,0.2,1.0,2.500000,0.12,inf,0.750
4,frozenset({Green}),frozenset({Red}),0.2,0.6,0.2,1.0,1.666667,0.08,inf,0.500
14,"frozenset({Green, White})",frozenset({Red}),0.2,0.6,0.2,1.0,1.666667,0.08,inf,0.500
7,frozenset({Orange}),frozenset({White}),0.2,0.7,0.2,1.0,1.428571,0.06,inf,0.375
8,frozenset({Green}),frozenset({White}),0.2,0.7,0.2,1.0,1.428571,0.06,inf,0.375


In [8]:
regras = rules.sort_values(by=['lift'], ascending=False)
rules_dataset = regras.drop(columns=['antecedent support', 'consequent support', 'conviction'])
rules_dataset.head()

,antecedents,consequents,support,confidence,lift,leverage,zhangs_metric
13,"frozenset({Red, White})",frozenset({Green}),0.2,0.5,2.500000,0.12,1.000
15,frozenset({Green}),"frozenset({Red, White})",0.2,1.0,2.500000,0.12,0.750
4,frozenset({Green}),frozenset({Red}),0.2,1.0,1.666667,0.08,0.500
14,"frozenset({Green, White})",frozenset({Red}),0.2,1.0,1.666667,0.08,0.500
7,frozenset({Orange}),frozenset({White}),0.2,1.0,1.428571,0.06,0.375


>## Comestibles

In [9]:
# Load and preprocess data set
comes = pd.read_csv('data/basket.csv')
comes.head()

,0,1,2,3,4,5,6,7,8,9,10
0,whole milk,pastry,salty snack,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sausage,whole milk,semi-finished bread,yogurt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,soda,pickled vegetables,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,canned beer,misc. beverages,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sausage,hygiene articles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
#tratando valores NaN
comes.replace(np.nan,0,inplace=True)
comes.head(10)

,0,1,2,3,4,5,6,7,8,9,10
0,whole milk,pastry,salty snack,0,0,0,0,0,0,0,0
1,sausage,whole milk,semi-finished bread,yogurt,0,0,0,0,0,0,0
2,soda,pickled vegetables,0,0,0,0,0,0,0,0,0
3,canned beer,misc. beverages,0,0,0,0,0,0,0,0,0
4,sausage,hygiene articles,0,0,0,0,0,0,0,0,0
5,sausage,whole milk,rolls/buns,0,0,0,0,0,0,0,0
6,whole milk,soda,0,0,0,0,0,0,0,0,0
7,frankfurter,soda,whipped/sour cream,0,0,0,0,0,0,0,0
8,frankfurter,curd,0,0,0,0,0,0,0,0,0
9,beef,white bread,0,0,0,0,0,0,0,0,0


In [11]:
# crear funcion lambda para mover los valosres x!=0 para uma lista
def removerTodosZerosLista(lista):
    return list(filter(lambda x: x!=0, lista))

In [12]:
#transformando en lista
lista_todas_transacciones = []
for index, row in comes.iterrows():
    lista_de_trans = row.values.tolist()
    lista_de_trans = removerTodosZerosLista(lista_de_trans)
    # adiciona a la lista general
    lista_todas_transacciones.append(lista_de_trans)
    
# visualiza lista
lista_todas_transacciones[0:10]

[['whole milk', 'pastry', 'salty snack'],
 ['sausage', 'whole milk', 'semi-finished bread', 'yogurt'],
 ['soda', 'pickled vegetables'],
 ['canned beer', 'misc. beverages'],
 ['sausage', 'hygiene articles'],
 ['sausage', 'whole milk', 'rolls/buns'],
 ['whole milk', 'soda'],
 ['frankfurter', 'soda', 'whipped/sour cream'],
 ['frankfurter', 'curd'],
 ['beef', 'white bread']]

In [13]:
# usando a biblioteca "mlxtend" para realizar preprocesamiento y ML.
#
Te = TransactionEncoder()
#
data_te = Te.fit(lista_todas_transacciones).transform(lista_todas_transacciones)
data_te #one-hot-encoding

array([[False, False, False, ...,  True, False, False],
       [False, False, False, ...,  True,  True, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])

In [14]:
# one-hot-encoding transformo todos los elementos comprados en columnas
Te.columns_[0:10]

['Instant food products',
 'UHT-milk',
 'abrasive cleaner',
 'artif. sweetener',
 'baby cosmetics',
 'bags',
 'baking powder',
 'bathroom cleaner',
 'beef',
 'berries']

In [15]:
# tamanho de la lista
len(Te.columns_)

167

In [16]:
# construyendo el DataFrame de los datos de las transacciones
T_data = pd.DataFrame(data_te, columns=Te.columns_)
T_data.head()

,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [17]:
# usando el algoritmo apriori
frecuentes_itemsets = apriori(T_data, min_support=0.003,use_colnames=True)
# ordenar de mayor para menor
frecuentes_itemsets.sort_values(by=['support'],ascending=False)
#
frecuentes_itemsets[['itemsets','support']][0:10]

,itemsets,support
0,frozenset({Instant food products}),0.004010
1,frozenset({UHT-milk}),0.021386
2,frozenset({baking powder}),0.008087
3,frozenset({beef}),0.033950
4,frozenset({berries}),0.021787
5,frozenset({beverages}),0.016574
6,frozenset({bottled beer}),0.045312
7,frozenset({bottled water}),0.060683
8,frozenset({brown bread}),0.037626
9,frozenset({butter}),0.035220


In [18]:
# importando y usando la asociacion de reglas
reglas = association_rules(frecuentes_itemsets,metric='confidence',min_threshold=0.001)
#
reglas_data = reglas.sort_values(by=['lift'],ascending=False)
reglas_data.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
6,frozenset({sausage}),frozenset({bottled beer}),0.060349,0.045312,0.003342,0.055371,1.222000,0.000607,1.010649,0.193337
7,frozenset({bottled beer}),frozenset({sausage}),0.045312,0.060349,0.003342,0.073746,1.222000,0.000607,1.014464,0.190292
73,frozenset({frankfurter}),frozenset({other vegetables}),0.037760,0.122101,0.005146,0.136283,1.116150,0.000536,1.016420,0.108146
72,frozenset({other vegetables}),frozenset({frankfurter}),0.122101,0.037760,0.005146,0.042146,1.116150,0.000536,1.004579,0.118536
184,frozenset({yogurt}),frozenset({sausage}),0.085879,0.060349,0.005748,0.066926,1.108986,0.000565,1.007049,0.107508


In [19]:
# lift > 1
rd = reglas_data.drop(['antecedent support','consequent support','leverage','conviction'], axis=1)
rd.head(10)

,antecedents,consequents,support,confidence,lift,zhangs_metric
6,frozenset({sausage}),frozenset({bottled beer}),0.003342,0.055371,1.222000,0.193337
7,frozenset({bottled beer}),frozenset({sausage}),0.003342,0.073746,1.222000,0.190292
73,frozenset({frankfurter}),frozenset({other vegetables}),0.005146,0.136283,1.116150,0.108146
72,frozenset({other vegetables}),frozenset({frankfurter}),0.005146,0.042146,1.116150,0.118536
184,frozenset({yogurt}),frozenset({sausage}),0.005748,0.066926,1.108986,0.107508
185,frozenset({sausage}),frozenset({yogurt}),0.005748,0.095238,1.108986,0.104587
129,frozenset({pastry}),frozenset({sausage}),0.003208,0.062016,1.027617,0.028341
128,frozenset({sausage}),frozenset({pastry}),0.003208,0.053156,1.027617,0.028601
178,frozenset({sausage}),frozenset({soda}),0.005948,0.098560,1.014975,0.015702
179,frozenset({soda}),frozenset({sausage}),0.005948,0.061253,1.014975,0.016341


> ### Análisis?

[1] Shmueli, G. bruce, P.C. Gedeck, P. Patel, N.R. Data mining for Business Analytics: Concepts, techniques and Application in Python. John Wiley & Sons, Inc. 2020.

---
<center><b>&copy;Jorge Zavaleta, 2024</b></center>